In [1]:
#fine tuning methods adapted from the transformers guide
#https://huggingface.co/docs/transformers/en/training
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [2]:


import pandas as pd
import re

#data['label'] = data['score'] >= 0

#data2 = Dataset.from_pandas(data)
#data['text'] = data['text'].apply(lambda x: re.sub(r'\[PET_BOUNDARY\]','',x))
#print(data['text'][0:10])
#data.to_csv('en_train_politeness_with_labels.csv')
from transformers import set_seed
from numpy.random import seed

#set_seed(0)


/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
text = 'text'
label = 'label'


In [4]:

#data = pd.read_csv('curated1.csv')
#data['text'] = data['text'].apply(lambda x: x.replace('[/PET_BOUNDARY]','[PET_BOUNDARY]'))
#data.to_csv('curated_cleaned.csv')
import torch

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
def auto_tokenize(dataset, tokenizer, text = 'text', i_d = 'Unnamed: 0', label = 'label', euph_status = 'euph_status', category = 'category', pet = 'PET', max_len=512):
    # load the tokenizer
    # Not finished, need to finish before using

    
    def tokenize_function(examples):# adapted from https://huggingface.co/docs/transformers/training
        a = tokenizer(examples, padding = False, truncation=False)
        if len(a['input_ids']) > max_len:
            a = False
            
        if a!=False:
            return tokenizer(examples, padding="max_length", max_length=max_len, truncation=True)
        else:
            return False
    output = {'input_ids':[], 'attention_mask': [], 'text': [], 'id': [], 'label':[]}
    
    if category in dataset.columns:
        output['category'] = []
    if euph_status in dataset.columns:
        output['status'] = []
    if pet in dataset.columns:
        output['pet'] = []
    
    for i in range(len(dataset)):
        y = tokenize_function(dataset.iloc[i][text])
        
        if y!=False:
            output['input_ids'].append(y['input_ids'])
            output['attention_mask'].append(y['attention_mask'])
            output['text'].append(dataset.iloc[i][text])
            output['id'].append(dataset.iloc[i][i_d])
            output['label'].append(dataset.iloc[i][label])
        
            if 'category' in output:
                if pd.notna(dataset[category].iloc[i]):
                    output['category'].append(dataset[category].iloc[i])
                else:
                    output['category'].append('')
            if 'status' in output:
                if pd.notna(dataset[euph_status].iloc[i]):
                    output['status'].append(dataset[euph_status].iloc[i])
                else:
                    output['status'].append('')
            if 'pet' in output:
                if pd.notna(dataset[pet].iloc[i]):
                    output['pet'].append(dataset[pet].iloc[i])
                else:
                    output['pet'].append('')
    
    return output

In [5]:
from transformers import AutoModelForSequenceClassification as be
from transformers import AutoTokenizer
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from transformers import EarlyStoppingCallback
from datasets import Dataset

In [6]:
def split(s, data_set, text):
    set_seed(s)
    bert = AutoTokenizer.from_pretrained('google-bert/bert-base-multilingual-cased')

    tokenized = auto_tokenize(data_set, text = text, max_len = 512, tokenizer = bert)


    data_euph = Dataset.from_dict(tokenized)
    
    split1 = data_euph.train_test_split(test_size = 0.3, seed = s)
    train_data = split1['train']
    split2 = split1['test'].train_test_split(test_size = 0.5, seed = s)
    val_data = split2['train']
    test_data = split2['test']

    
    return [train_data, val_data, test_data]

def convert(s, data_set, text = 'TEXT'):
    set_seed(s)
    bert = AutoTokenizer.from_pretrained('google-bert/bert-base-multilingual-cased')

    tokenized = auto_tokenize(data_set, text = 'TEXT', i_d = 'ID', pet = 'PET', category = 'CATEGORY', euph_status = 'EUPH_STATUS', label = 'LABEL', max_len = 512, tokenizer = bert)


    data_euph = Dataset.from_dict(tokenized)
    
    return data_euph

In [7]:
output_dir = 'output'

training_args = TrainingArguments(output_dir=output_dir,
                                  num_train_epochs=20,
                                  learning_rate= 1e-5,
                                  per_device_train_batch_size=16,
                                  per_device_eval_batch_size=16,
                                  #gradient_accumulation_steps = 4,
                                  #gradient_checkpointing = True,
                                  #eval_accumulation_steps = 1,
                                  logging_strategy = 'epoch',
                                  logging_first_step = True,
                                  save_strategy = 'epoch',
                                  load_best_model_at_end = True,
                                  metric_for_best_model = 'f1',
                                  eval_strategy = "epoch",
                                  report_to = "none",
                                  bf16 = True)

def seq_fine_tune_2(s, model, training_args, train_data, val_data, test_data, text, label):
    #s is seed number
    set_seed(s)
    
    


    
    def compute_metrics(p):
        logits, labels = p
        pred = logits[0]
        pred = np.argmax(pred, axis=1)
        accuracy = accuracy_score(y_true=labels, y_pred=pred)
        recall = recall_score(y_true=labels, y_pred=pred, average='macro')
        precision = precision_score(y_true=labels, y_pred=pred, average='macro')
        f1 = f1_score(y_true=labels, y_pred=pred, average='macro')
        return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}
    
    

    trainer = Trainer(model = model.cuda(), args = training_args, train_dataset = train_data, eval_dataset = val_data, compute_metrics = compute_metrics, callbacks = [EarlyStoppingCallback(early_stopping_patience= 5 )])
    
    trainer.train()
    
    n_epochs = trainer.state.epoch
    
    return [model, train_data, val_data, test_data, n_epochs]

In [8]:
from sklearn.linear_model import LogisticRegression

def logistic_reg_test(s,model, train_data, test_data, name):
    def batch(data, size):
        if len(data) < size:
            return [data]
        else:
            start = 0
            end = start + size
            batches = []
            while start < len(data):
                batches.append(data[start:end])
                start = end
                if start + size <= len(data):
                    end = start + size
                else:
                    end = len(data)
            return batches
    
    ti = batch(train_data['input_ids'],64)
    ta = batch(train_data['attention_mask'],64)
    tei = batch(test_data['input_ids'],64)
    tea = batch(test_data['attention_mask'],64)
    
    train_input = []
    train_am = []
    test_input = []
    test_am = []
    for i in ti:
        train_input.append(torch.Tensor(i).to(torch.int64))
    for i in tei:
        test_input.append(torch.Tensor(i).to(torch.int64))
    for i in ta:
        train_am.append(torch.Tensor(i).to(torch.int64))
    for i in tea:
        test_am.append(torch.Tensor(i).to(torch.int64))
        
    
    embeddings = model(train_input[0].cuda(), train_am[0].cuda()).hidden_states[-1][:,0,:].tolist()

    for i in range(1,len(train_input)):
        embeddings = embeddings + model(train_input[i].cuda(), train_am[i].cuda()).hidden_states[-1][:,0,:].tolist()

    
    inputs = np.array(embeddings)
    #print(inputs.shape)
    labels = np.array(train_data[label])
    
    embeddings_test = model(test_input[0].cuda(), test_am[0].cuda()).hidden_states[-1][:,0,:].tolist()
    for i in range(1,len(test_input)):
        embeddings_test = embeddings_test + model(test_input[i].cuda(), test_am[i].cuda()).hidden_states[-1][:,0,:].tolist()

    
    lr = LogisticRegression(random_state=s, penalty = 'l2', solver = 'sag', max_iter = 1000)
    lr.fit(inputs, labels)
    
    testing_predictions = lr.predict(embeddings_test)
    
    if name!=None:
        table = {'predicted': testing_predictions}
        for i in test_data.column_names:
            table[i] = test_data[i]
        table = pd.DataFrame(table)
        table.to_csv('tables_bert2/'+str(s)+'_'+name+'_table.csv')
    
    return [accuracy_score(test_data['label'], testing_predictions), precision_score(test_data['label'], testing_predictions), recall_score(test_data['label'], testing_predictions), f1_score(test_data['label'], testing_predictions), f1_score(test_data['label'], testing_predictions, average = 'macro')]

def single_ft(seed_start, seed_end,training_args, data_csv, text, label, name):
    data = pd.read_csv(data_csv)
    
    datacsv = re.sub(r'\.csv','',data_csv)
    
    
    for i in range(seed_start, seed_end):
        set_seed(i)
        
        results = {'train_data': [], 'test_data': [], 'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'f1_macro': [], 'seed': []}
        splits = split(i, data, text)
        splits = split(i, data, text)
        
        
        mm = be.from_pretrained('google-bert/bert-base-multilingual-cased', num_labels = 2, output_hidden_states = True)
        mm.cuda()
        #print(splits[0]['input_ids'][0])
        ft2 = seq_fine_tune_2(i,mm,training_args,splits[0],splits[1],splits[2],text,label)
        
        #t = test(ft1[0], ft1[3])
        t = logistic_reg_test(i,ft2[0],splits[0],splits[2],datacsv+'_'+name)
        
        results['train_data'].append(datacsv)
        results['test_data'].append(datacsv)
        results['accuracy'].append(t[0])
        results['precision'].append(t[1])
        results['recall'].append(t[2])
        results['f1'].append(t[3])
        results['f1_macro'].append(t[4])
        results['seed'].append(i)
        

        results = pd.DataFrame(results)
        
        results.to_csv('f1s_bert/single_'+re.sub(r'\.csv','',data_csv)+str(i)+'.csv')        
    return

def pre(seed_start, seed_end, data_csv, text, label, name):
    data = pd.read_csv(data_csv)
    
    datacsv = re.sub(r'\.csv','',data_csv)
    
    
    for i in range(seed_start, seed_end):
        set_seed(i)
        
        results = {'train_data': [], 'test_data': [], 'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'f1_macro': [], 'seed': []}
        

        mm = be.from_pretrained('google-bert/bert-base-multilingual-cased', num_labels = 2, output_hidden_states = True)
        mm.cuda()

        splits = split(i, data, text)
        #print(splits[0]['input_ids'][0])
        #t = test(ft1[0], ft1[3])
        t = logistic_reg_test(i,mm,splits[0],splits[2],'pre_'+name)
        
        results['train_data'].append('pretrained')
        results['test_data'].append(datacsv)
        results['accuracy'].append(t[0])
        results['precision'].append(t[1])
        results['recall'].append(t[2])
        results['f1'].append(t[3])
        results['f1_macro'].append(t[4])
        results['seed'].append(i)
        

        results = pd.DataFrame(results)
        results.to_csv('f1s_bert/pre_'+re.sub(r'\.csv','',data_csv)+str(i)+'.csv')


In [9]:
def cross_task(seed_start, seed_end, training_args, data_csv, text, label, data_csv_2, text2, label2, name):
    data = pd.read_csv(data_csv)
    data2 = pd.read_csv(data_csv_2)
    
    datacsv = re.sub(r'\.csv','',data_csv)
    datacsv2 = re.sub(r'\.csv','',data_csv_2)
    
    
    for i in range(seed_start, seed_end):
        set_seed(i)
        
        results = {'train_data': [], 'train_data_2': [], 'test_data': [], 'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'f1_macro': [], 'n_epochs_1': [], 'n_epochs_2': [], 'seed': []}

        
        splits = split(i, data, text)
        splits2 = split(i, data2, text2)
        
        mm = be.from_pretrained('google-bert/bert-base-multilingual-cased', num_labels = 2, output_hidden_states = True)
        mm.cuda()
        ft1 = seq_fine_tune_2(i,mm,training_args,splits[0],splits[1],splits[2],text,label)

        
        t = logistic_reg_test(i,ft1[0],splits[0],splits[2],None)
        
        results['train_data'].append(datacsv)
        results['train_data_2'].append('na')
        results['test_data'].append(datacsv)
        results['accuracy'].append(t[0])
        results['precision'].append(t[1])
        results['recall'].append(t[2])
        results['f1'].append(t[3])
        results['f1_macro'].append(t[4])
        results['n_epochs_1'].append(ft1[4])
        results['n_epochs_2'].append('')
        results['seed'].append(i)
        
        t = logistic_reg_test(i,ft1[0],splits2[0],splits2[2],datacsv+'_'+name)
        
        results['train_data'].append(datacsv)
        results['train_data_2'].append('na')
        results['test_data'].append(datacsv2)
        results['accuracy'].append(t[0])
        results['precision'].append(t[1])
        results['recall'].append(t[2])
        results['f1'].append(t[3])
        results['f1_macro'].append(t[4])
        results['n_epochs_1'].append(ft1[4])
        results['n_epochs_2'].append('')
        results['seed'].append(i)
        

    
        results = pd.DataFrame(results)
        
        print('generating CSV containing results: "crosstask_tests_'+re.sub(r'\.csv','',data_csv)+'_'+re.sub(r'\.csv','',data_csv_2)+str(i)+'.csv"')
        results.to_csv('f1s_bert/crosstask_tests_'+re.sub(r'\.csv','',data_csv)+'_'+re.sub(r'\.csv','',data_csv_2)+str(i)+'.csv')

        
    return mm

<h3>Adjust code below</h3>

In [10]:
def cross_task_test(seed_start, seed_end, training_args, data_csv, text, label, data_csv_2, text2, label2, test_train, test_test, name):
    data = pd.read_csv(data_csv)
    data2 = pd.read_csv(data_csv_2)
    t_tr = pd.read_csv(test_train)
    t_te = pd.read_csv(test_test)
    
    datacsv = re.sub(r'\.csv','',data_csv)
    datacsv2 = re.sub(r'\.csv','',data_csv_2)
    ttr = re.sub(r'\.csv','',test_train)
    tte = re.sub(r'\.csv','',test_test)
    
    
    for i in range(seed_start, seed_end):
        set_seed(i)
        
        results = {'train_data': [], 'train_data_2': [], 'test_data': [], 'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'f1_macro': [], 'n_epochs_1': [], 'n_epochs_2': [], 'seed': []}

        
        splits = split(i, data, text)
        splits2 = split(i, data2, text2)#this does nothing except anchor the determinism so it matches the other mode
        
        mm = be.from_pretrained('google-bert/bert-base-multilingual-cased', num_labels = 2, output_hidden_states = True)
        mm.cuda()
        ft1 = seq_fine_tune_2(i,mm,training_args,splits[0],splits[1],splits[2],text,label)
        
        ft1[0].save_pretrained('xlmr_models/'+ datacsv + '_'+str(i))
        
        t = logistic_reg_test(i,ft1[0],splits[0],splits[2],None)
        
        results['train_data'].append(datacsv)
        results['train_data_2'].append('na')
        results['test_data'].append(datacsv)
        results['accuracy'].append(t[0])
        results['precision'].append(t[1])
        results['recall'].append(t[2])
        results['f1'].append(t[3])
        results['f1_macro'].append(t[4])
        results['n_epochs_1'].append(ft1[4])
        results['n_epochs_2'].append('')
        results['seed'].append(i)
        
        t_tra = convert(i, t_tr)
        t_tes = convert(i, t_te)
        
        t = logistic_reg_test(i,ft1[0],t_tra,t_tes,datacsv+name)
        
        results['train_data'].append(datacsv)
        results['train_data_2'].append('na')
        results['test_data'].append('euph')
        results['accuracy'].append(t[0])
        results['precision'].append(t[1])
        results['recall'].append(t[2])
        results['f1'].append(t[3])
        results['f1_macro'].append(t[4])
        results['n_epochs_1'].append(ft1[4])
        results['n_epochs_2'].append('')
        results['seed'].append(i)
        

    
        results = pd.DataFrame(results)
        
        print('generating CSV containing results: "crosstask_tests_'+re.sub(r'\.csv','',data_csv)+'_'+re.sub(r'\.csv','',data_csv_2)+str(i)+'.csv"')
        results.to_csv('f1s_bert/crosstask_tests_'+re.sub(r'\.csv','',data_csv)+'_'+re.sub(r'\.csv','',data_csv_2)+str(i)+'.csv')

        
    return mm

In [11]:
result = cross_task(0,10,training_args,'trofi_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
result = cross_task(0,10,training_args,'trofi_boundary_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')

result = cross_task(0,10,training_args,'trofi_boundary_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')

result = cross_task(0,10,training_args,'trofi_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')

"""
#result = pre(0,10,'en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
result = pre(0,10,'en_sometimes_euph_1900.csv','text','label','_boundary')
#result = single_ft(0,10,training_args,'en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
result = single_ft(0,10,training_args,'en_sometimes_euph_1900.csv','text','label','_boundary')

#result = cross_task(0,10,training_args,'en_sometimes_euph_no_boundary_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')
result = cross_task(0,10,training_args,'en_sometimes_euph_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')

result = cross_task(0,10,training_args,'magpie_no_boundary_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')
#result = cross_task(0,10,training_args,'magpie_no_boundary_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
result = cross_task(0,10,training_args,'movie_pos_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')
#result = cross_task(0,10,training_args,'movie_pos_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')

#result = cross_task(0,10,training_args,'books_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')
#result = cross_task(0,10,training_args,'books_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
#result = cross_task(0,10,training_args,'trofi_1900.csv','sentence','label','en_sometimes_euph_1900.csv','text','label','_boundary')

result = cross_task(7,10,training_args,'trofi_1900.csv','sentence','label','en_sometimes_euph_1900.csv','text','label','_boundary')
result = cross_task(0,10,training_args,'polite_1900.csv','sentence','label','en_sometimes_euph_1900.csv','text','label','_boundary')

#result = cross_task(0,10,training_args,'polite_1900.csv','sentence','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
#result = cross_task(0,10,training_args,'trofi_boundary_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
result = cross_task(0,10,training_args,'trofi_boundary_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')



result = cross_task(0,10,training_args,'sens_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')
#result = cross_task(0,10,training_args,'sens_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')

#result = cross_task(0,10,training_args,'trofi_1900.csv','sentence','label','en_sometimes_euph_1900.csv','text','label','_boundary')


result = cross_task(0,10,training_args,'idem_1900.csv','sentence','label','en_sometimes_euph_1900.csv','text','label','_boundary')

#result = cross_task(0,10,training_args,'trofi_1900.csv','sentence','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')

#result = cross_task(8,10,training_args,'idem_1900.csv','sentence','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
result = cross_task(0,10,training_args,'magpie_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')
result = cross_task(0,10,training_args,'multi_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')

#result = cross_task(0,10,training_args,'magpie_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
#result = cross_task(0,10,training_args,'multi_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')
"""


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.665800,0.591380,0.768421,0.772086,0.772356,0.768418
2,0.553600,0.467274,0.810526,0.809430,0.810012,0.809680
3,0.441200,0.422368,0.838596,0.837770,0.839275,0.838148
4,0.355100,0.423672,0.821053,0.825535,0.825535,0.821053
5,0.278400,0.422073,0.842105,0.841755,0.843583,0.841825
6,0.222200,0.464724,0.828070,0.832591,0.832591,0.828070
7,0.177500,0.478590,0.838596,0.841481,0.842395,0.838579
8,0.143100,0.510002,0.831579,0.832685,0.834299,0.831477
9,0.126300,0.632123,0.796491,0.807287,0.803179,0.796288
10,0.088600,0.561013,0.824561,0.823721,0.825163,0.824074


/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarnin

generating CSV containing results: "crosstask_tests_trofi_1900_en_sometimes_euph_no_boundary_19000.csv"


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.682800,0.641156,0.666667,0.666667,0.664677,0.664669
2,0.561400,0.547699,0.726316,0.739766,0.730519,0.724522
3,0.433100,0.499598,0.778947,0.779590,0.779838,0.778936
4,0.351300,0.490173,0.800000,0.807029,0.796853,0.797446
5,0.280600,0.503149,0.803509,0.803571,0.802673,0.802963
6,0.218900,0.547643,0.800000,0.803534,0.802007,0.799911
7,0.164100,0.562851,0.803509,0.805750,0.801588,0.802221
8,0.138700,0.605638,0.810526,0.813651,0.812414,0.810468
9,0.113100,0.617855,0.800000,0.800620,0.798752,0.799199
10,0.088000,0.659181,0.807018,0.806714,0.806865,0.806780


/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/pohw/.conda/envs/Euphemisms/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarnin

generating CSV containing results: "crosstask_tests_trofi_1900_en_sometimes_euph_19009.csv"


"\n#result = pre(0,10,'en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')\nresult = pre(0,10,'en_sometimes_euph_1900.csv','text','label','_boundary')\n#result = single_ft(0,10,training_args,'en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')\nresult = single_ft(0,10,training_args,'en_sometimes_euph_1900.csv','text','label','_boundary')\n\n#result = cross_task(0,10,training_args,'en_sometimes_euph_no_boundary_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')\nresult = cross_task(0,10,training_args,'en_sometimes_euph_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')\n\nresult = cross_task(0,10,training_args,'magpie_no_boundary_1900.csv','text','label','en_sometimes_euph_1900.csv','text','label','_boundary')\n#result = cross_task(0,10,training_args,'magpie_no_boundary_1900.csv','text','label','en_sometimes_euph_no_boundary_1900.csv','text','label','_noboundary')\nresult = cr